In [ ]:
import requests
import numpy as np
import rasterio
from rasterio.merge import merge
from rasterio.mask import mask
from rasterio.transform import from_origin
import pyproj
from shapely.geometry import shape, mapping
from shapely.ops import transform as shp_transform
import os, json

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Referer": "https://www.ign.es/",
}
from pathlib import Path

RAW_DIR     = Path("../../data/raw/03_pendiente_dem")
DELIM_DIR   = Path("../../data/raw/delimitations")
RAW_DIR.mkdir(parents=True, exist_ok=True)

TILE_DIR    = RAW_DIR / "tiles_huesca"
TILE_SIZE   = 10000
MOSAIC_TIF  = os.path.join(TILE_DIR, "_mosaic_5m.tif")
SLOPE_5M    = os.path.join(TILE_DIR, "_slope_5m.tif")
OUTPUT_TIF  = RAW_DIR / "pendiente_huesca_provincia.tif"
CHUNK       = 2048          # filas procesadas a la vez; bájalo a 1024 si el PC se queda sin RAM
OVERLAP     = 2             # píxeles de solape para el gradiente en bordes
os.makedirs(TILE_DIR, exist_ok=True)

GEOJSON_HUESCA = DELIM_DIR / "Huesca_Delimitacion.geojson"

# ── 1. Cargar GeoJSON y reproyectar a EPSG:25830 ──────────────────────────
with open(GEOJSON_HUESCA) as f:
    geojson = json.load(f)

if geojson["type"] == "FeatureCollection":
    geom = shape(geojson["features"][0]["geometry"])
elif geojson["type"] == "Feature":
    geom = shape(geojson["geometry"])
else:
    geom = shape(geojson)

project  = pyproj.Transformer.from_crs("EPSG:4326", "EPSG:25830", always_xy=True).transform
geom_utm = shp_transform(project, geom)

X_MIN, Y_MIN, X_MAX, Y_MAX = [int(v) for v in geom_utm.bounds]
print(f"Bbox UTM25830: ({X_MIN}, {Y_MIN}) → ({X_MAX}, {Y_MAX})")

# ── 2. Descargar tiles MDT a 5m (paralelo, con reanudación) ──────────────
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

MAX_WORKERS = 8   # nº de descargas simultáneas — sube/baja según cómo responda el servidor

xs    = list(range(X_MIN, X_MAX, TILE_SIZE))
ys    = list(range(Y_MIN, Y_MAX, TILE_SIZE))
tile_coords = [(x0, y0) for x0 in xs for y0 in ys]
total = len(tile_coords)

print_lock = threading.Lock()
progress = {"done": 0}

def descargar_tile(coords):
    x0, y0 = coords
    x1, y1 = x0 + TILE_SIZE, y0 + TILE_SIZE
    tile_path = os.path.join(TILE_DIR, f"tile_{x0}_{y0}.tif")

    if os.path.exists(tile_path):
        with print_lock:
            progress["done"] += 1
            print(f"[{progress['done']}/{total}] CACHED ({x0},{y0})")
        return tile_path

    qs = (
        "service=WCS&version=2.0.1&request=GetCoverage"
        "&coverageId=Elevacion25830_5"
        "&format=image/tiff"
        f"&subset=x({x0},{x1})"
        f"&subset=y({y0},{y1})"
        "&subsettingCrs=http://www.opengis.net/def/crs/EPSG/0/25830"
        "&outputCrs=http://www.opengis.net/def/crs/EPSG/0/25830"
    )
    try:
        r = requests.get(
            f"https://servicios.idee.es/wcs-inspire/mdt?{qs}",
            headers=HEADERS, timeout=120,
        )
        with print_lock:
            progress["done"] += 1
        if "tiff" not in r.headers.get("Content-Type", ""):
            with print_lock:
                print(f"[{progress['done']}/{total}] SKIP ({x0},{y0}): {r.text[:100]}")
            return None
        with open(tile_path, "wb") as fout:
            fout.write(r.content)
        with print_lock:
            print(f"[{progress['done']}/{total}] OK ({x0},{y0})")
        return tile_path
    except Exception as e:
        with print_lock:
            progress["done"] += 1
            print(f"[{progress['done']}/{total}] ERROR ({x0},{y0}): {e}")
        return None

tile_files = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(descargar_tile, c) for c in tile_coords]
    for future in as_completed(futures):
        result = future.result()
        if result:
            tile_files.append(result)

print(f"\nDescarga completa: {len(tile_files)}/{total} tiles")

# ── 3. Merge de tiles → GeoTIFF comprimido en disco ──────────────────────
if not os.path.exists(MOSAIC_TIF):
    print(f"\nMerging {len(tile_files)} tiles → {MOSAIC_TIF} ...")
    datasets = [rasterio.open(f) for f in tile_files]
    mosaic, transform = merge(datasets)
    profile = datasets[0].profile.copy()
    profile.update(
        driver="GTiff",
        width=mosaic.shape[2],
        height=mosaic.shape[1],
        transform=transform,
        compress="deflate",
        tiled=True,
        blockxsize=512,
        blockysize=512,
        bigtiff="YES",
    )
    for ds in datasets:
        ds.close()
    with rasterio.open(MOSAIC_TIF, "w", **profile) as dst:
        dst.write(mosaic)
    del mosaic   # liberar RAM inmediatamente
    print("Mosaic guardado.")
else:
    print(f"Mosaic ya existe: {MOSAIC_TIF}")

# ── 4. Calcular pendiente por CHUNKS y escribir a disco ───────────────────
print(f"\nCalculando pendiente por chunks de {CHUNK} filas (overlap={OVERLAP})...")

with rasterio.open(MOSAIC_TIF) as src:
    H, W   = src.height, src.width
    nodata = src.nodata if src.nodata is not None else -9999
    prof5  = src.profile.copy()

prof5.update(
    dtype="float32",
    nodata=-9999,
    compress="deflate",
    tiled=True,
    blockxsize=512,
    blockysize=512,
    bigtiff="YES",
)

with rasterio.open(SLOPE_5M, "w", **prof5) as dst:
    with rasterio.open(MOSAIC_TIF) as src:
        row = 0
        while row < H:
            # Ventana con solape
            r_start = max(row - OVERLAP, 0)
            r_end   = min(row + CHUNK + OVERLAP, H)

            win     = rasterio.windows.Window(0, r_start, W, r_end - r_start)
            dem     = src.read(1, window=win).astype("float32")
            dem[dem == nodata] = np.nan

            dy, dx  = np.gradient(dem, 5.0)
            slope_chunk = np.degrees(np.arctan(np.sqrt(dx**2 + dy**2)))
            slope_chunk = np.nan_to_num(slope_chunk, nan=-9999).astype("float32")

            # Recortar el solape para escribir solo las filas "buenas"
            top_cut    = row - r_start          # cuántas filas solapadas hay arriba
            bottom_cut = r_end - (row + CHUNK)  # cuántas filas solapadas hay abajo
            actual_end = row + CHUNK if row + CHUNK < H else H
            n_rows     = actual_end - row

            write_data = slope_chunk[top_cut : top_cut + n_rows, :]
            win_write  = rasterio.windows.Window(0, row, W, n_rows)
            dst.write(write_data, 1, window=win_write)

            pct = min((row + CHUNK) / H * 100, 100)
            print(f"  Filas {row}–{actual_end-1} / {H}  ({pct:.0f}%)")
            row += CHUNK

print(f"Pendiente 5m guardada: {SLOPE_5M}")

# ── 5. Resample 5m → 25m por bloques ─────────────────────────────────────
print("\nResampleando 5m → 25m (average, bloques)...")
FACTOR = 5

with rasterio.open(SLOPE_5M) as src:
    H5, W5  = src.height, src.width
    tf5     = src.transform
    H25     = H5 // FACTOR
    W25     = W5 // FACTOR

    new_tf  = from_origin(tf5.c, tf5.f, tf5.a * FACTOR, abs(tf5.e) * FACTOR)
    prof25  = src.profile.copy()
    prof25.update(
        width=W25, height=H25, transform=new_tf,
        dtype="float32", nodata=-9999,
        compress="deflate", tiled=True,
        blockxsize=512, blockysize=512,
        bigtiff="YES",
    )

    SLOPE_25_TMP = os.path.join(TILE_DIR, "_slope_25m_tmp.tif")
    with rasterio.open(SLOPE_25_TMP, "w", **prof25) as dst:
        row5 = 0
        while row5 < H25 * FACTOR:
            n5       = min(CHUNK * FACTOR, H25 * FACTOR - row5)
            n25      = n5 // FACTOR
            win5     = rasterio.windows.Window(0, row5, W5, n5)
            data5    = src.read(1, window=win5).astype("float32")

            # Recortar a múltiplo exacto de FACTOR
            data5    = data5[:n25 * FACTOR, :W25 * FACTOR]
            data25   = (
                data5
                .reshape(n25, FACTOR, W25, FACTOR)
                .mean(axis=(1, 3))
                .astype("float32")
            )
            # Propagar nodata
            nd_mask  = (
                (data5 == -9999)
                .reshape(n25, FACTOR, W25, FACTOR)
                .any(axis=(1, 3))
            )
            data25[nd_mask] = -9999

            row25    = row5 // FACTOR
            win25    = rasterio.windows.Window(0, row25, W25, n25)
            dst.write(data25, 1, window=win25)

            pct = min((row5 + n5) / (H25 * FACTOR) * 100, 100)
            print(f"  Filas 5m {row5}–{row5+n5-1}  ({pct:.0f}%)")
            row5 += n5

print(f"Resample completado: {SLOPE_25_TMP}")

# ── 6. Clip al límite provincial ──────────────────────────────────────────
print("\nClipping al límite provincial...")
with rasterio.open(SLOPE_25_TMP) as src:
    clipped, clipped_tf = mask(src, [mapping(geom_utm)], crop=True, nodata=-9999)
    clipped_profile = src.profile.copy()
    clipped_profile.update(
        width=clipped.shape[2], height=clipped.shape[1],
        transform=clipped_tf, compress="deflate",
    )

os.unlink(SLOPE_25_TMP)

# ── 7. Guardar resultado final ────────────────────────────────────────────
with rasterio.open(OUTPUT_TIF, "w", **clipped_profile) as dst:
    dst.write(clipped)

valid = clipped[0][clipped[0] != -9999]
print(f"\n✓ Guardado: {OUTPUT_TIF}")
print(f"  Resolución  : 25m × 25m")
print(f"  Dimensiones : {clipped_profile['width']} × {clipped_profile['height']} px")
print(f"  Tamaño mem. : {clipped.nbytes / 1e6:.1f} MB")
print(f"  Min={valid.min():.1f}°  Max={valid.max():.1f}°  Media={valid.mean():.1f}°")
print(f"  Píxeles válidos: {len(valid):,}  |  Nodata: {(clipped[0]==-9999).sum():,}")

Bbox UTM25830: (669547, 4581688) → (810631, 4754656)
[1/270] CACHED (669547,4581688)
[2/270] CACHED (669547,4591688)
[3/270] CACHED (669547,4601688)
[4/270] CACHED (669547,4611688)
[5/270] CACHED (669547,4621688)
[6/270] CACHED (669547,4641688)
[7/270] CACHED (669547,4631688)
[8/270] CACHED (669547,4661688)
[9/270] CACHED (669547,4691688)
[10/270] CACHED (669547,4651688)
[11/270] CACHED (669547,4681688)
[12/270] CACHED (669547,4711688)
[13/270] CACHED (669547,4701688)
[14/270] CACHED (669547,4671688)
[15/270] CACHED (669547,4721688)
[16/270] CACHED (669547,4731688)
[17/270] CACHED (669547,4741688)
[18/270] CACHED (669547,4751688)
[19/270] CACHED (679547,4581688)
[20/270] CACHED (679547,4591688)
[21/270] CACHED (679547,4601688)
[22/270] CACHED (679547,4611688)
[23/270] CACHED (679547,4621688)
[24/270] CACHED (679547,4631688)
[25/270] CACHED (679547,4651688)
[26/270] CACHED (679547,4661688)
[27/270] CACHED (679547,4641688)
[28/270] CACHED (679547,4721688)
[29/270] CACHED (679547,4731688)

In [ ]:
import numpy as np
import rasterio
import pandas as pd
import pydeck as pdk
from pyproj import Transformer
from pathlib import Path

RAW_DIR = Path("../../data/raw/03_pendiente_dem")
MAP_DIR = Path("../../docs/map/03_pendiente_dem")
RAW_DIR.mkdir(parents=True, exist_ok=True)
MAP_DIR.mkdir(parents=True, exist_ok=True)

# --- Leer TIF ---
with rasterio.open(RAW_DIR / "pendiente_huesca_provincia.tif") as src:
    slope = src.read(1).astype("float32")
    nodata = src.nodata or -9999
    slope[slope == nodata] = np.nan
    transform = src.transform

# --- Submuestrear ---
step = 25

slope_sub = slope[::step, ::step]
rows, cols = slope_sub.shape

xs = np.array([transform.c + (j * step + step/2) * transform.a for j in range(cols)])
ys = np.array([transform.f + (i * step + step/2) * transform.e for i in range(rows)])
X, Y = np.meshgrid(xs, ys)

mask = ~np.isnan(slope_sub)
x_flat = X[mask]
y_flat = Y[mask]
s_flat = slope_sub[mask]

# --- UTM25830 → WGS84 ---
tr = Transformer.from_crs("EPSG:25830", "EPSG:4326", always_xy=True)
lon, lat = tr.transform(x_flat, y_flat)

# --- Color por clase de pendiente (RGBA) ---
def slope_color(s):
    if s < 5:   return [34,  139, 34,  200]   # verde oscuro — plano
    if s < 15:  return [144, 238, 144, 200]   # verde claro  — suave
    if s < 30:  return [255, 255, 100, 220]   # amarillo     — moderado
    if s < 45:  return [255, 140, 0,   230]   # naranja      — fuerte
    return              [180, 0,   0,   255]   # rojo         — extremo

# --- Categoría textual por clase de pendiente (misma lógica que slope_color) ---
def slope_category(s):
    if s < 5:   return "Plano"
    if s < 15:  return "Suave"
    if s < 30:  return "Moderado"
    if s < 45:  return "Fuerte"
    return "Extremo"

colors = np.array([slope_color(s) for s in s_flat], dtype=np.uint8)
categorias = np.array([slope_category(s) for s in s_flat])

df = pd.DataFrame({
    "lon":       lon,
    "lat":       lat,
    # float64 ANTES de redondear: evita el ruido de precisión de float32
    # (si no, en el tooltip sale algo tipo 21.100000381469727 en vez de 21.1)
    "pendiente": np.round(s_flat.astype("float64"), 1),
    "categoria": categorias,
    "elevacion": s_flat * 160,   # escala visual Z (ajusta el *160 a tu gusto)
    "r": colors[:, 0],
    "g": colors[:, 1],
    "b": colors[:, 2],
    "a": colors[:, 3],
})

print(f"Puntos: {len(df):,}")

# --- ColumnLayer: columnas 3D de altura proporcional a la pendiente ---
layer = pdk.Layer(
    "ColumnLayer",
    data=df,
    get_position=["lon", "lat"],
    get_elevation="elevacion",
    get_fill_color=["r", "g", "b", "a"],
    radius=200,          # radio de cada columna en metros — baja si step es pequeño
    elevation_scale=1,
    pickable=True,
    auto_highlight=True,
)

view = pdk.ViewState(
    longitude=-0.08,
    latitude=42.10,      # centrado en Huesca provincia
    zoom=8,
    min_zoom=7,
    max_zoom=12,
    pitch=50,
    bearing=-20,
)

# Tooltip: deck.gl solo sustituye {nombre_columna} tal cual está en los datos,
# NO soporta formato estilo Python (":.1f"), por eso el redondeo se hace arriba,
# en el propio DataFrame, y aquí solo se referencia la columna ya limpia.
tooltip = {
    "html": "<b>Pendiente:</b> {pendiente}°<br/><b>Clase:</b> {categoria}",
    "style": {"backgroundColor": "#111", "color": "white", "fontSize": "12px"},
}

r = pdk.Deck(
    layers=[layer],
    initial_view_state=view,
    tooltip=tooltip,
    map_style="https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json",
)

output_html = MAP_DIR / "mapa_pendiente_3d_huesca.html"
r.to_html(str(output_html))

# --- Leyenda fija inyectada en el HTML final ---
# pydeck no tiene leyenda nativa, así que se agrega un <div> propio justo antes
# de </body>. Los colores RGB coinciden exactamente con los de slope_color().
legend_html = """
<div style="position:fixed; bottom:20px; left:20px; background:#111; color:white;
            padding:12px 16px; border-radius:8px; font-family:sans-serif; font-size:13px;
            z-index:9999; box-shadow:0 2px 8px rgba(0,0,0,0.5);">
  <b>Pendiente</b>
  <div style="margin-top:6px;"><span style="display:inline-block;width:14px;height:14px;background:rgb(34,139,34);margin-right:6px;border-radius:2px;"></span>&lt; 5° — Plano</div>
  <div style="margin-top:4px;"><span style="display:inline-block;width:14px;height:14px;background:rgb(144,238,144);margin-right:6px;border-radius:2px;"></span>5–15° — Suave</div>
  <div style="margin-top:4px;"><span style="display:inline-block;width:14px;height:14px;background:rgb(255,255,100);margin-right:6px;border-radius:2px;"></span>15–30° — Moderado</div>
  <div style="margin-top:4px;"><span style="display:inline-block;width:14px;height:14px;background:rgb(255,140,0);margin-right:6px;border-radius:2px;"></span>30–45° — Fuerte</div>
  <div style="margin-top:4px;"><span style="display:inline-block;width:14px;height:14px;background:rgb(180,0,0);margin-right:6px;border-radius:2px;"></span>&gt; 45° — Extremo</div>
</div>
"""

# ─────────────────────────────────────────────────────────
# Título, fuente y escala/norte (misma vestimenta que el resto
# de mapas del proyecto) — no se toca la leyenda ni la paleta
# ─────────────────────────────────────────────────────────
import math

INITIAL_ZOOM = view.zoom
INITIAL_LAT  = view.latitude

m_per_px = 156543.03392 * math.cos(math.radians(INITIAL_LAT)) / (2 ** INITIAL_ZOOM)
scale_km_options = [1, 2, 5, 10, 20, 25, 50]
target_px = 110
scale_km = min(scale_km_options, key=lambda k: abs((k * 1000 / m_per_px) - target_px))
scale_px = round((scale_km * 1000) / m_per_px)

overlay_extra_html = f"""
<style>
  .map-panel {{
    font-family: 'Inter','Segoe UI',sans-serif;
    background: rgba(18,20,24,0.82);
    backdrop-filter: blur(6px);
    color: #f2f2f2;
    border: 1px solid rgba(255,255,255,0.10);
    border-radius: 10px;
    box-shadow: 0 4px 18px rgba(0,0,0,0.35);
  }}
</style>

<div class="map-panel" style="position:fixed;top:18px;left:18px;padding:14px 20px;
            max-width:380px;z-index:9999;">
  <div style="font-size:16px;font-weight:600;letter-spacing:0.2px;">
    Pendiente del terreno — Huesca (vista 3D)
  </div>
  <div style="font-size:12px;color:rgba(255,255,255,0.6);margin-top:3px;">
    Columnas proporcionales a la pendiente, MDT 5 m
  </div>
</div>

<div style="position:fixed;top:18px;right:18px;font-family:'Inter','Segoe UI',sans-serif;
            font-size:10.5px;color:rgba(255,255,255,0.45);z-index:9999;text-align:right;">
  Fuente: Modelo Digital del Terreno 5 m (IGN) — pendiente derivada<br/>
  TFM — Análisis de idoneidad, planta de biometano
</div>

<div class="map-panel" style="position:fixed;bottom:26px;right:18px;padding:12px 18px;
            z-index:9999;">
  <div style="display:flex;align-items:center;gap:14px;">
    <svg width="30" height="30" viewBox="0 0 30 30">
      <line x1="15" y1="26" x2="15" y2="6" stroke="#f2f2f2" stroke-width="1.6"/>
      <polygon points="15,3 11,12 19,12" fill="#f2f2f2"/>
      <text x="15" y="0" font-size="9" fill="#f2f2f2" text-anchor="middle"
            font-family="Inter,sans-serif" font-weight="600">N</text>
    </svg>
    <div>
      <div style="width:{scale_px}px;height:0;border-top:2px solid #f2f2f2;position:relative;">
        <div style="position:absolute;left:0;top:-7px;width:1.5px;height:6px;background:#f2f2f2;"></div>
        <div style="position:absolute;right:0;top:-7px;width:1.5px;height:6px;background:#f2f2f2;"></div>
      </div>
      <div style="font-size:11px;color:rgba(255,255,255,0.75);margin-top:4px;">
        ≈ {scale_km} km <span style="color:rgba(255,255,255,0.45);">(zoom inicial, vista en planta)</span>
      </div>
    </div>
  </div>
</div>
"""

html_content = output_html.read_text(encoding="utf-8")
html_content = html_content.replace("</body>", legend_html + overlay_extra_html + "</body>")
output_html.write_text(html_content, encoding="utf-8")

print(f"Abre: {output_html}")

In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import LightSource
from matplotlib.patches import Patch
from matplotlib.patches import FancyArrow
from pathlib import Path

RAW_DIR = Path("../../data/raw/03_pendiente_dem")
MAP_DIR = Path("../../docs/map/03_pendiente_dem")
RAW_DIR.mkdir(parents=True, exist_ok=True)
MAP_DIR.mkdir(parents=True, exist_ok=True)

# --- Leer TIF ---
with rasterio.open(RAW_DIR / "pendiente_huesca_provincia.tif") as src:
    slope = src.read(1).astype("float32")
    nodata = src.nodata or -9999
    slope[slope == nodata] = np.nan
    px_size_m = src.res[0]   # tamaño de píxel en metros (EPSG:25830), para la escala

# --- Hillshade ---
ls = LightSource(azdeg=315, altdeg=45)
hillshade = ls.hillshade(slope, vert_exag=2)

# --- Colormap por clases ---
cmap = mcolors.ListedColormap([
    "#228B22",   # < 5°   plano
    "#90EE90",   # 5-15°  suave
    "#FFFF64",   # 15-30° moderado
    "#FF8C00",   # 30-45° fuerte
    "#B40000",   # > 45°  extremo
])
bounds = [0, 5, 15, 30, 45, 90]
norm = mcolors.BoundaryNorm(bounds, cmap.N)

# --- Plot ---
fig, ax = plt.subplots(figsize=(12, 10))

ax.imshow(hillshade, cmap="gray", interpolation="bilinear")
ax.imshow(slope, cmap=cmap, norm=norm, alpha=0.6, interpolation="bilinear")

ax.set_title("Mapa de Pendientes — Provincia de Huesca", fontsize=14, fontweight="bold")
ax.axis("off")

leyenda = [
    Patch(color="#228B22", label="< 5° — Plano"),
    Patch(color="#90EE90", label="5–15° — Suave"),
    Patch(color="#FFFF64", label="15–30° — Moderado"),
    Patch(color="#FF8C00", label="30–45° — Fuerte"),
    Patch(color="#B40000", label="> 45° — Extremo"),
]
ax.legend(handles=leyenda, loc="lower right", fontsize=10, framealpha=0.9)

# ─────────────────────────────────────────────────────────
# Norte, escala y fuente — no se toca paleta/título/leyenda
# ─────────────────────────────────────────────────────────
h, w = slope.shape

# --- Rosa de los vientos (arriba a la izquierda) ---
arrow_x, arrow_y = w * 0.045, h * 0.10
arrow_len = h * 0.045
ax.annotate(
    "N",
    xy=(arrow_x, arrow_y - arrow_len),
    xytext=(arrow_x, arrow_y),
    ha="center", va="center", fontsize=11, fontweight="bold", color="white",
    arrowprops=dict(facecolor="white", edgecolor="white", width=3, headwidth=11, headlength=10),
)

# --- Barra de escala (abajo a la izquierda, en metros/píxel reales) ---
scale_km_options = [1, 2, 5, 10, 15, 20, 25]
target_px = w * 0.16
scale_km = min(scale_km_options, key=lambda k: abs((k * 1000 / px_size_m) - target_px))
scale_px = (scale_km * 1000) / px_size_m

bar_x0 = w * 0.045
bar_y  = h * 0.965
ax.plot([bar_x0, bar_x0 + scale_px], [bar_y, bar_y], color="white", linewidth=2.5,
        solid_capstyle="butt")
ax.plot([bar_x0, bar_x0], [bar_y - h * 0.006, bar_y + h * 0.006], color="white", linewidth=2.5)
ax.plot([bar_x0 + scale_px, bar_x0 + scale_px], [bar_y - h * 0.006, bar_y + h * 0.006],
        color="white", linewidth=2.5)
ax.text(bar_x0 + scale_px / 2, bar_y - h * 0.018, f"{scale_km} km",
        ha="center", va="bottom", fontsize=9.5, color="white", fontweight="medium")

# --- Fuente de datos (esquina inferior derecha) ---
fig.text(
    0.985, 0.012,
    "Fuente: Modelo Digital del Terreno 5 m (IGN) — pendiente derivada\n"
    "TFM — Análisis de idoneidad, planta de biometano",
    ha="right", va="bottom", fontsize=7.5, color="#444444", linespacing=1.4,
)

plt.tight_layout()
plt.savefig(MAP_DIR / "mapa_pendiente_huesca.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
print(f"Guardado: {MAP_DIR / 'mapa_pendiente_huesca.png'}")